# 01 — Build catalog

One-time ETL from the Becker prediction-market-analysis Parquet dump into
NautilusTrader's ParquetDataCatalog.

In [ ]:
# Notebook is location-agnostic: walk up to find the project root
# (the dir containing pyproject.toml) and chdir there before any imports.
import os, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
if (_p / "pyproject.toml").exists():
    os.chdir(_p)
del _p


In [ ]:
from datetime import datetime, timezone
from src.layer1_research.backtesting.data.models import MarketFilter
from src.layer1_research.backtesting.data.loaders.becker_parquet import BeckerParquetLoader
from src.layer1_research.backtesting.data.catalog import build_catalog

## Config

In [ ]:
DATA_PATH = "../prediction-market-analysis/data"   # Becker repo (sibling checkout)
CATALOG_PATH = "data/catalog"
MIN_VOLUME = 100.0
DATE_START = datetime(2023, 3, 1, tzinfo=timezone.utc)  # trades data begins 2023-03
LIMIT = 100   # cap markets for fast ETL; remove for full load

## Load

In [ ]:
loader = BeckerParquetLoader(DATA_PATH)
filters = MarketFilter(min_volume=MIN_VOLUME, date_start=DATE_START)

### Preview which markets will be loaded

In [ ]:
markets = loader.load_markets(filters=filters)
print(f"Matched {len(markets)} markets; will load first {LIMIT}")
for m in markets[:10]:
    print(f"  {m.market_id[:16]}...  {m.question[:60]}")

### Build catalog

In [ ]:
result = build_catalog(loader, CATALOG_PATH, filters=filters, limit=LIMIT)
print(f"\nMarkets:     {result.markets_loaded}")
print(f"Trades:      {result.trades_loaded:,}")
print(f"Instruments: {result.instruments_created}")
print(f"Path:        {result.catalog_path}")